*italicized text*
 # Finetuning a pre-trained model

 In this notebook, we will demonstrate how to fine-tune a pre-trained model on a custom dataset for specific tasks.


In [ ]:
!pip install transformers datasets torch

: 

# Fine-Tuning Pre-trained Models
While the pre-trained models provided by Hugging Face are powerful, you may want to fine-tune them for a specific task or dataset.

Fine-tuning involves taking a pre-trained model and training it further on your own data. This can improve the model’s performance for specific use cases.

For this section, we’ll load the IMDB dataset (which contains movie reviews) and fine-tune a pre-trained model for sentiment classification.

### Load Dataset
We'll use Hugging Face's datasets library to load the IMDB dataset.

Datasets from the dataset library often come with pre-defined splits of the data, such as `train` and `test` sets.

It is possible to filter or slice datasets to focus on specific subsets of the data, using the `select` method.

### GPU Usage

Note that if you run this notebook on your own device, it will be rather slow. We recommend using the T4 GPU on Google Colab to run this.

In [1]:
from datasets import load_dataset
import torch
from transformers import AutoTokenizer
from transformers import Trainer, TrainingArguments
from transformers import AutoModelForSequenceClassification


device = 0 if torch.cuda.is_available() else -1

dataset = load_dataset("imdb")
train_dataset = (
    dataset["train"].shuffle(seed=42).select(range(100))
)  # Using a subset for quick fine-tuning
test_dataset = dataset["test"].shuffle(seed=42).select(range(100))

/home/efeoz/nlp/natural-language-processing/assignment3/src/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Tokenize the Dataset
The dataset needs to be tokenized before it can be fed into the model. Tokenization converts the text data into numerical format (tokens) that the model can process.

We'll use the `AutoTokenizer` class from HuggingFace to tokenize the data. The `AutoTokenizer` class automatically selects the appropriate tokenizer for the model based on the `model_name`.

Tokenization or transformation of the dataset can be done using the `map` method, which applies a function to all the elements of the dataset. This is easily done by defining a function that tokenizes the text data and then applying it to the dataset. When `batched=True`, the function will be applied to batches of data, which can improve performance by applying the function in parallel.

In [4]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)


def tokenize_function(examples):
    # print(examples["text"][0])
    return tokenizer(examples["text"], padding="max_length", truncation=True)


tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

### Load a Pre-trained Model
Now that the data is tokenized, we'll load a pre-trained model that we'll fine-tune for sentiment classification.

We'll use distilbert-base-uncased for this task.

We need to import `AutoModelForSequenceClassification` for that. The key feature of this class is that it adds a classification head on top of the pre-trained transformer model to allow it to classify sequences into one or more categories (e.g., positive/negative sentiment, spam/ham, etc.). The `from_pretrained` method loads the pre-trained model with the specified configuration. The `num_labels` parameter specifies the number of labels in the classification task (binary in this case).

In [5]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(
    device
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 754.41it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### Set Up the Trainer
Hugging Face provides the Trainer class to help with the training and fine-tuning of models. We need to set up the trainer by providing the model, training arguments, and the datasets.


In [6]:
training_args = TrainingArguments(
    eval_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
)

### Fine-tune the Model
Now that the trainer is set up, we can start the fine-tuning process.

Run the following cell to fine-tune the model.

In [7]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.694494,0.674961
2,0.640448,0.654403
3,0.558782,0.628483
4,0.489807,0.607146
5,0.429624,0.598773


Writing model shards: 100%|██████████| 1/1 [00:04<00:00,  4.25s/it]


TrainOutput(global_step=65, training_loss=0.5626310201791617, metrics={'train_runtime': 54.1183, 'train_samples_per_second': 9.239, 'train_steps_per_second': 1.201, 'total_flos': 66233699328000.0, 'train_loss': 0.5626310201791617, 'epoch': 5.0})

### Evaluate the Model
After training, we can evaluate the model's performance on the test set.

In [10]:
eval_results = trainer.evaluate()
print(f"Evaluation Results: {eval_results}")

RuntimeError: on_train_begin must be called before on_evaluate

### Try out model

In [11]:
input_string = "I really liked this tutorial!"

# Tokenize the input string
inputs = tokenizer(input_string, return_tensors="pt").to(device)

# Get predictions (logits)
with torch.no_grad():  # Disable gradient computation since we're just doing inference
    outputs = model(**inputs)
    logits = outputs.logits

predicted_label = torch.argmax(logits, dim=1).item()


print(f"Predicted label: {predicted_label}")

Predicted label: 1
